# Surper_GCG 12B 流水线 Notebook

这个 notebook 是重构后 `12B` 流水线在 Colab 上的唯一入口。

只通过 `run_pipeline.py` 运行，不再混用旧实验入口。

当前可运行的主线：
- `t0_t1_bootstrap`
- `t0_t2_bootstrap`

不要再把旧的 `L17/L23` 包装脚本当成 `12B` 主线。

## 1. 挂载 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆或更新仓库

In [ ]:
%cd /content
import os

REPO_DIR = '/content/Surper_GCG'
REPO_URL = 'https://github.com/awaa-col/Surper_GCG_ResearchWorkFlow.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists: {REPO_DIR}')

%cd /content/Surper_GCG
!git pull --ff-only

## 3. 安装项目

In [ ]:
%cd /content/Surper_GCG
!pip install -e .

## 4. 运行参数配置

正式运行前，只需要改这个 cell。

In [ ]:
import os

HF_TOKEN = ''
RUN_NAME = 'gemma3_12b_main'
MODEL = 'google/gemma-3-12b-it'
RESUME = True
N_TRAIN = 32
N_EVAL = 8
MAX_NEW_TOKENS = 96
SEED = 42

DRIVE_ROOT = '/content/drive/MyDrive/surper_gcg_pipeline_runs'
LOCAL_RESULTS_ROOT = '/content/Surper_GCG/results/pipeline_runs'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/{RUN_NAME}'
LOCAL_RUN_DIR = f'{LOCAL_RESULTS_ROOT}/{RUN_NAME}'

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

RESUME_FLAG = '--resume' if RESUME else ''

print({
    'RUN_NAME': RUN_NAME,
    'MODEL': MODEL,
    'RESUME': RESUME,
    'N_TRAIN': N_TRAIN,
    'N_EVAL': N_EVAL,
    'MAX_NEW_TOKENS': MAX_NEW_TOKENS,
    'SEED': SEED,
    'DRIVE_RUN_DIR': DRIVE_RUN_DIR,
    'LOCAL_RUN_DIR': LOCAL_RUN_DIR,
})

## 5. 查看当前可用的 Preset 和 Stage

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py --list-presets
!python run_pipeline.py --list-stages

## 6. 可选：从 Drive 恢复已有进度

如果 Colab 重启过，而你想继续之前的 run，就先执行这个 cell。

In [ ]:
import os
import shutil

os.makedirs(LOCAL_RESULTS_ROOT, exist_ok=True)

if not os.path.exists(DRIVE_RUN_DIR):
    print(f'No saved run found on Drive: {DRIVE_RUN_DIR}')
else:
    if os.path.exists(LOCAL_RUN_DIR):
        shutil.rmtree(LOCAL_RUN_DIR)
    shutil.copytree(DRIVE_RUN_DIR, LOCAL_RUN_DIR)
    print(f'Restored run: {DRIVE_RUN_DIR} -> {LOCAL_RUN_DIR}')

## 7. `T0-T1` Dry Run

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t1_bootstrap \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 8. 运行 `T0-T1`

这一步是当前 `12B` 主线的标准起点。

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t1_bootstrap \
  --run-name "$RUN_NAME" \
  $RESUME_FLAG \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 9. 检查 `T0-T1` 产物

在继续做 gate discovery 之前，先检查这些产物。

In [ ]:
!ls -lah "$LOCAL_RUN_DIR"
!test -f "$LOCAL_RUN_DIR/exp11_review_pack.jsonl" && echo 'Found exp11_review_pack.jsonl'
!test -f "$LOCAL_RUN_DIR/exp00_diagnosis.json" && echo 'Found exp00_diagnosis.json'
!test -f "$LOCAL_RUN_DIR/pipeline_manifest.json" && echo 'Found pipeline_manifest.json'
!test -f "$LOCAL_RUN_DIR/pipeline_stage_summary.md" && echo 'Found pipeline_stage_summary.md'

## 10. `T0-T2` Dry Run

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t2_bootstrap \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 11. 运行 `T0-T2`

只有在你确认 `T0-T1` 结果没问题之后，再执行这一步。

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t2_bootstrap \
  --run-name "$RUN_NAME" \
  $RESUME_FLAG \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 12. 检查最终产物

In [ ]:
!ls -lah /content/Surper_GCG/results/pipeline_runs
!ls -lah "$LOCAL_RUN_DIR"

## 13. 把当前 Run 同步回 Drive

In [ ]:
import os
import shutil

os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.exists(DRIVE_RUN_DIR):
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_RUN_DIR, DRIVE_RUN_DIR)
print(f'Saved run to Drive: {DRIVE_RUN_DIR}')

## 14. 可选：导出 Zip 备份

In [ ]:
!zip -r "/content/drive/MyDrive/${RUN_NAME}_pipeline_results.zip" "$LOCAL_RUN_DIR"